# Evidence convergence and binning sensitivity

This notebook holds each simulated periodogram fixed while changing the number of prior draws, the bin width, and the evidence estimator. This is a paired experiment: differences cannot be blamed on a different stochastic realization of the star.

In [ ]:
import numpy as np
from asterodetect import (
    AsteroScaleSamples, AstrophysicalInjectionFactory, ObservationModel,
    build_detection_study, run_sensitivity_study,
)
from asterodetect.asteroscale import ASTERO_SCALE_PARAMETERS

In [ ]:
rng = np.random.default_rng(123)
latent = rng.normal(size=512)
values = {name: np.ones(512) for name in ASTERO_SCALE_PARAMETERS}
values.update(
    numax=3100 * np.exp(0.025 * latent), dnu=135.1 * np.exp(0.018 * latent),
    FWHM_env=950 * np.exp(0.08 * latent), A_env=2.1 * np.exp(0.12 * latent),
    A_gran=55 * np.exp(-0.10 * latent),
    b_gran_low=760 * np.exp(0.025 * latent),
    b_gran_high=2850 * np.exp(0.025 * latent),
)
samples = AsteroScaleSamples(values)

We generate one noise case, one granulation case, and two oscillation cases. The latter use suppressed and expected amplitudes. For a scientific run, increase both the number of injected realizations and the inference repeats.

In [ ]:
factory = AstrophysicalInjectionFactory(samples, duration_days=27.4, cadence_seconds=120)
cases = build_detection_study(
    {'white_noise': [0.1], 'duration_days': [27.4], 'dilution': [1.0]},
    factory, oscillation_amplitudes=[0.3, 1.0], repeats=1, seed=10,
)
[(case.truth, case.metadata['amplitude_scale']) for case in cases]

In [ ]:
study = run_sensitivity_study(
    cases,
    draw_counts=[32, 128],
    dnu_scales=[0.5, 1.0, 2.0],
    estimators=['prior', 'adaptive'],
    repeats=2,
    seed=11,
    observation=ObservationModel(integration_time_seconds=120),
)
study.summaries()

Interpret the diagnostics together. The oscillation-probability scatter should shrink with more draws; the ESS fraction should not collapse; and the log-evidence standard error should decrease. Comparing classification accuracy alone can hide an unstable evidence estimate. The three bin scales test the trade-off between averaging over the mode comb and retaining the broad envelope shape. The paired estimator comparison then asks whether adaptive sampling improves ESS and evidence error without materially shifting the inferred probabilities.

In [ ]:
rows = [
    (s.draws, s.dnu_scale, s.estimator, s.oscillation_probability_std,
     s.minimum_median_ess_fraction,
     s.maximum_median_log_evidence_standard_error,
     s.classification_accuracy)
    for s in study.summaries()
]
rows, study.estimator_comparisons()